# Uji Kelayakan Fine-Tuning LoRA Whisper pada Perangkat Lokal

Notebook ini menjawab satu pertanyaan: **varian Whisper mana yang masih bisa dilatih dengan LoRA di mesinmu, dan berapa lama.**

Cara kerjanya: untuk tiap varian, notebook memasang modul LoRA sesuai konfigurasi Liu, Yang dan Qu (2024), menjalankan beberapa langkah latih dengan data tiruan, lalu mengukur puncak pemakaian memori dan waktu per langkah. Hasilnya diekstrapolasi ke ukuran data latihmu yang sebenarnya.

Data tiruan dipakai karena bentuk masukan Whisper tetap: spektrogram log-Mel selalu setara 30 detik audio (Polok, et al., 2026), jadi pemakaian memori dan waktu per langkah tidak bergantung pada isi audionya. Yang diukur di sini kelayakan komputasi, bukan akurasi.

**Urutan pemakaian:** jalankan Bagian 1 sampai 5 berurutan. Bagian 6 dan 7 opsional.

**Peringatan unduhan.** Varian besar berukuran besar: `medium` sekitar 3 GB, `large-v3-turbo` sekitar 3,2 GB, `large-v3` sekitar 6 GB. Sesuaikan `VARIAN_DIUJI` di Bagian 2 kalau kuota atau ruang penyimpananmu terbatas.

## Bagian 1 — Pemasangan dan pemeriksaan lingkungan

In [1]:
# Jalankan sekali saja. Hapus tanda komentar bila paket belum terpasang.
# !pip install -q torch transformers peft accelerate psutil
# Untuk pengoptimal 8-bit (opsional, hanya CUDA):
# !pip install -q bitsandbytes

In [2]:
import os, sys, time, math, gc, platform, json
import torch, psutil

def info_lingkungan():
    d = {}
    d["sistem_operasi"] = f"{platform.system()} {platform.release()}"
    d["python"] = sys.version.split()[0]
    d["torch"] = torch.__version__
    d["cpu"] = platform.processor() or platform.machine()
    d["inti_fisik"] = psutil.cpu_count(logical=False)
    d["inti_logis"] = psutil.cpu_count(logical=True)
    d["ram_total_gb"] = round(psutil.virtual_memory().total / 1024**3, 1)
    d["ram_tersedia_gb"] = round(psutil.virtual_memory().available / 1024**3, 1)
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        d["gpu"] = p.name
        d["vram_gb"] = round(p.total_memory / 1024**3, 1)
        d["cuda"] = torch.version.cuda
        d["kapabilitas"] = f"{p.major}.{p.minor}"
        d["dukung_bf16"] = torch.cuda.is_bf16_supported()
    else:
        d["gpu"] = None
    return d

ENV = info_lingkungan()
for k, v in ENV.items():
    print(f"{k:20s}: {v}")

if ENV["gpu"] is None:
    print("\n[!] Tidak ada GPU CUDA terdeteksi.")
    print("    Pelatihan LoRA akan berjalan di CPU dan sangat lambat.")
    print("    Notebook tetap dapat dijalankan; hasilnya menjadi bukti terukur")
    print("    bahwa pelatihan lokal tidak layak, dan itu temuan yang sah untuk Bab 3.")

sistem_operasi      : Windows 11
python              : 3.12.4
torch               : 2.4.1+cu121
cpu                 : AMD64 Family 25 Model 117 Stepping 2, AuthenticAMD
inti_fisik          : 8
inti_logis          : 16
ram_total_gb        : 13.8
ram_tersedia_gb     : 1.0
gpu                 : NVIDIA GeForce RTX 3050 6GB Laptop GPU
vram_gb             : 6.0
cuda                : 12.1
kapabilitas         : 8.6
dukung_bf16         : True


## Bagian 2 — Konfigurasi

Ubah nilai di sel ini agar sesuai rancangan penelitianmu. Nilai bawaan mengikuti Liu, Yang dan Qu (2024) untuk LoRA dan angka pembagian data 65 : 10 : 25 dari 12 jam rekaman.

In [3]:
# --- Varian yang diuji -------------------------------------------------------
# Urut dari kecil ke besar. Notebook berhenti pada varian yang gagal
# kalau HENTI_SETELAH_GAGAL bernilai True.
VARIAN_DIUJI = [
    ("tiny",           "openai/whisper-tiny"),
    ("base",           "openai/whisper-base"),
    ("small",          "openai/whisper-small"),
    ("medium",         "openai/whisper-medium"),
    ("large-v3-turbo", "openai/whisper-large-v3-turbo"),
    ("large-v3",       "openai/whisper-large-v3"),
]
HENTI_SETELAH_GAGAL = True

# --- Konfigurasi LoRA (Liu, Yang dan Qu, 2024) -------------------------------
LORA_R          = 32
LORA_ALPHA      = 64
LORA_DROPOUT    = 0.05
TARGET_MODULES  = ["q_proj", "v_proj"]   # posisi query dan value pada modul atensi

# --- Konfigurasi pelatihan ---------------------------------------------------
BATCH_SIZE        = 4      # Liu, Yang dan Qu (2024) memakai 4 pada A100 80 GB
GRAD_ACCUM        = 1      # naikkan bila batch harus diturunkan
PANJANG_LABEL     = 128    # perkiraan token per segmen 30 detik ujaran bahasa Indonesia
GRADIENT_CHECKPOINTING = False   # aktifkan pada Bagian 6 bila kehabisan memori
PENGOPTIMAL_8BIT  = False        # butuh bitsandbytes, hanya CUDA

# --- Ukuran data latih yang sebenarnya ---------------------------------------
JAM_DATA_LATIH        = 7.7    # 65% dari 12 jam
RERATA_DETIK_SEGMEN   = 20.0   # segmen hasil pemotongan berbasis VAD, di bawah batas 30 detik
EPOCH                 = 35     # Liu, Yang dan Qu (2024). Adila, et al. (2024) memakai 10.000 langkah

# --- Pengukuran --------------------------------------------------------------
LANGKAH_PEMANASAN = 3
LANGKAH_UKUR      = 8

PERANGKAT = "cuda" if torch.cuda.is_available() else "cpu"
if PERANGKAT == "cuda":
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    DTYPE = torch.float32

print(f"Perangkat        : {PERANGKAT}")
print(f"Tipe data model  : {DTYPE}")
print(f"Batch efektif    : {BATCH_SIZE * GRAD_ACCUM}")

n_segmen      = int(JAM_DATA_LATIH * 3600 / RERATA_DETIK_SEGMEN)
langkah_epoch = math.ceil(n_segmen / (BATCH_SIZE * GRAD_ACCUM))
total_langkah = langkah_epoch * EPOCH
print(f"\nPerkiraan jumlah segmen latih : {n_segmen:,}")
print(f"Langkah per epoch             : {langkah_epoch:,}")
print(f"Total langkah ({EPOCH} epoch)      : {total_langkah:,}")

Perangkat        : cuda
Tipe data model  : torch.bfloat16
Batch efektif    : 4

Perkiraan jumlah segmen latih : 1,386
Langkah per epoch             : 347
Total langkah (35 epoch)      : 12,145


## Bagian 3 — Taksiran memori secara analitik

Taksiran kasar sebelum apa pun diunduh. Berguna untuk menyaring varian yang jelas tidak muat, sehingga waktu tidak terbuang mengunduh model 6 GB yang pasti gagal.

Komponen pemakaian memori saat melatih LoRA:

- **Bobot beku**, ukuran terbesar tetapi tidak menerima gradien.
- **Parameter LoRA**, sangat kecil. Liu, Yang dan Qu (2024) melaporkan 7,7 juta parameter, yaitu 0,67% dari model 1,15 miliar.
- **Gradien dan status pengoptimal**, hanya untuk parameter LoRA, jadi ikut kecil.
- **Aktivasi**, bagian yang sebenarnya menentukan. Nilainya naik seiring ukuran batch dan bisa ditekan dengan gradient checkpointing.

In [4]:
PARAMETER_JUTA = {
    "tiny": 39, "base": 74, "small": 244, "medium": 769,
    "large-v3-turbo": 809, "large-v3": 1550,
}

def taksir_memori(nama, dtype_bytes, batch):
    p = PARAMETER_JUTA[nama] * 1e6
    bobot_beku   = p * dtype_bytes
    p_lora       = p * 0.0067                      # nisbah dari Liu, Yang dan Qu (2024)
    lora_fp32    = p_lora * 4
    gradien      = p_lora * 4
    status_adamw = p_lora * 8                      # momen pertama dan kedua
    # Aktivasi ditaksir kasar dari jumlah parameter, panjang urutan encoder 1500,
    # dan ukuran batch. Faktor pengalinya empiris, bukan turunan teoretis.
    aktivasi     = p * dtype_bytes * batch * 0.55
    total = bobot_beku + lora_fp32 + gradien + status_adamw + aktivasi
    return {
        "bobot beku": bobot_beku, "LoRA": lora_fp32, "gradien": gradien,
        "status pengoptimal": status_adamw, "aktivasi": aktivasi, "TOTAL": total,
    }

db = 2 if DTYPE in (torch.float16, torch.bfloat16) else 4
batas = ENV.get("vram_gb") if PERANGKAT == "cuda" else ENV["ram_tersedia_gb"]

print(f"Taksiran pemakaian memori, batch {BATCH_SIZE}, presisi {db*8}-bit")
print(f"Kapasitas tersedia: {batas} GB\n")
print(f"{'Varian':<16}{'Bobot':>9}{'Aktivasi':>10}{'TOTAL':>9}   Status")
print("-" * 62)
for nama, _ in VARIAN_DIUJI:
    m = taksir_memori(nama, db, BATCH_SIZE)
    gb = lambda x: x / 1024**3
    status = "muat" if gb(m["TOTAL"]) < batas * 0.85 else ("ketat" if gb(m["TOTAL"]) < batas else "TIDAK MUAT")
    print(f"{nama:<16}{gb(m['bobot beku']):>8.2f}G{gb(m['aktivasi']):>9.2f}G{gb(m['TOTAL']):>8.2f}G   {status}")

print("\nTaksiran ini kasar. Angka yang dipakai di Bab 3 adalah hasil pengukuran Bagian 5.")

Taksiran pemakaian memori, batch 4, presisi 16-bit
Kapasitas tersedia: 6.0 GB

Varian              Bobot  Aktivasi    TOTAL   Status
--------------------------------------------------------------
tiny                0.07G     0.16G    0.24G   muat
base                0.14G     0.30G    0.45G   muat
small               0.45G     1.00G    1.48G   muat
medium              1.43G     3.15G    4.66G   muat
large-v3-turbo      1.51G     3.32G    4.90G   muat
large-v3            2.89G     6.35G    9.39G   TIDAK MUAT

Taksiran ini kasar. Angka yang dipakai di Bab 3 adalah hasil pengukuran Bagian 5.


## Bagian 4 — Fungsi pengukuran

In [5]:
from transformers import WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model

def bersihkan():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def puncak_memori_gb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1024**3
    return psutil.Process().memory_info().rss / 1024**3

def uji_varian(nama, model_id, batch=None, grad_ckpt=None, optim8bit=None):
    """Memasang LoRA, menjalankan beberapa langkah latih, mengembalikan hasil ukur."""
    batch     = BATCH_SIZE if batch is None else batch
    grad_ckpt = GRADIENT_CHECKPOINTING if grad_ckpt is None else grad_ckpt
    optim8bit = PENGOPTIMAL_8BIT if optim8bit is None else optim8bit

    hasil = {"varian": nama, "batch": batch, "grad_ckpt": grad_ckpt, "optim8bit": optim8bit}
    bersihkan()
    t0 = time.time()

    try:
        model = WhisperForConditionalGeneration.from_pretrained(model_id, torch_dtype=DTYPE)
        model.config.forced_decoder_ids = None
        model.config.use_cache = False
        n_mels = model.config.num_mel_bins        # 80 pada varian awal, 128 pada large-v3 dan turbo
        hasil["n_mels"] = n_mels
        hasil["waktu_muat_detik"] = round(time.time() - t0, 1)

        cfg = LoraConfig(
            r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
            target_modules=TARGET_MODULES, bias="none",
        )
        model = get_peft_model(model, cfg)

        # Parameter LoRA dipertahankan pada presisi 32-bit agar pembaruannya stabil.
        for _, p in model.named_parameters():
            if p.requires_grad:
                p.data = p.data.float()

        terlatih = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total    = sum(p.numel() for p in model.parameters())
        hasil["parameter_terlatih"] = terlatih
        hasil["parameter_total"]    = total
        hasil["persen_terlatih"]    = round(100 * terlatih / total, 3)

        model.to(PERANGKAT)
        if grad_ckpt:
            model.gradient_checkpointing_enable()
            try:
                model.enable_input_require_grads()
            except AttributeError:
                pass

        if optim8bit:
            import bitsandbytes as bnb
            opt = bnb.optim.AdamW8bit([p for p in model.parameters() if p.requires_grad], lr=1e-5)
        else:
            opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-5)

        if DTYPE == torch.float16:
            try:
                skaler = torch.amp.GradScaler("cuda")
            except (AttributeError, TypeError):
                skaler = torch.cuda.amp.GradScaler()
        else:
            skaler = None
        v = model.config.vocab_size

        def satu_langkah():
            fitur = torch.randn(batch, n_mels, 3000, dtype=DTYPE, device=PERANGKAT)
            label = torch.randint(0, v - 100, (batch, PANJANG_LABEL), device=PERANGKAT)
            with torch.autocast(device_type=PERANGKAT, dtype=DTYPE, enabled=(PERANGKAT == "cuda")):
                keluaran = model(input_features=fitur, labels=label)
                rugi = keluaran.loss
            if skaler is not None:
                skaler.scale(rugi).backward(); skaler.step(opt); skaler.update()
            else:
                rugi.backward(); opt.step()
            opt.zero_grad(set_to_none=True)

        for _ in range(LANGKAH_PEMANASAN):
            satu_langkah()
        if torch.cuda.is_available():
            torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()

        t1 = time.time()
        for _ in range(LANGKAH_UKUR):
            satu_langkah()
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        hasil["detik_per_langkah"] = round((time.time() - t1) / LANGKAH_UKUR, 3)
        hasil["puncak_memori_gb"]  = round(puncak_memori_gb(), 2)
        hasil["status"] = "BERHASIL"

    except (torch.cuda.OutOfMemoryError if torch.cuda.is_available() else RuntimeError) as e:
        hasil["status"] = "MEMORI HABIS"
        hasil["pesan"]  = str(e).split("\n")[0][:160]
    except Exception as e:
        hasil["status"] = "GAGAL"
        hasil["pesan"]  = f"{type(e).__name__}: {str(e)[:160]}"
    finally:
        model = None
        opt = None
        bersihkan()

    return hasil


def ekstrapolasi(detik_per_langkah):
    """Menerjemahkan waktu per langkah menjadi total waktu pelatihan."""
    total_detik = detik_per_langkah * total_langkah
    return {
        "jam_total": round(total_detik / 3600, 2),
        "jam_per_epoch": round(detik_per_langkah * langkah_epoch / 3600, 2),
    }

print("Fungsi siap.")

c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fungsi siap.


## Bagian 5 — Menjalankan pengukuran

Sel ini mengunduh dan menguji tiap varian berurutan. Jalankan sekali; hasilnya disimpan ke `hasil_uji_lora.json` supaya tidak perlu diulang.

In [6]:
HASIL = []
for nama, model_id in VARIAN_DIUJI:
    print(f"\n{'='*66}\nMenguji {nama}")
    h = uji_varian(nama, model_id)
    HASIL.append(h)

    if h["status"] == "BERHASIL":
        e = ekstrapolasi(h["detik_per_langkah"])
        h.update(e)
        print(f"  parameter terlatih : {h['parameter_terlatih']:,} ({h['persen_terlatih']}%)")
        print(f"  pita Mel           : {h['n_mels']}")
        print(f"  puncak memori      : {h['puncak_memori_gb']} GB")
        print(f"  detik per langkah  : {h['detik_per_langkah']}")
        print(f"  taksiran waktu     : {e['jam_per_epoch']} jam/epoch, {e['jam_total']} jam total")
    else:
        print(f"  {h['status']}: {h.get('pesan','')}")
        if HENTI_SETELAH_GAGAL:
            print("\n  Varian yang lebih besar dilewati karena pasti ikut gagal.")
            print("  Coba penanganan pada Bagian 6.")
            break

with open("hasil_uji_lora.json", "w") as f:
    json.dump({"lingkungan": ENV, "hasil": HASIL}, f, indent=2, default=str)
print("\nHasil disimpan ke hasil_uji_lora.json")


Menguji tiny


c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative\.venv\Lib\site-packages\transformers\models\whisper\modeling_whisper.py:599: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


  parameter terlatih : 589,824 (1.538%)
  pita Mel           : 80
  puncak memori      : 0.76 GB
  detik per langkah  : 0.116
  taksiran waktu     : 0.01 jam/epoch, 0.39 jam total

Menguji base
  parameter terlatih : 1,179,648 (1.599%)
  pita Mel           : 80
  puncak memori      : 1.19 GB
  detik per langkah  : 0.228
  taksiran waktu     : 0.02 jam/epoch, 0.77 jam total

Menguji small
  parameter terlatih : 3,538,944 (1.443%)
  pita Mel           : 80
  puncak memori      : 2.99 GB
  detik per langkah  : 0.723
  taksiran waktu     : 0.07 jam/epoch, 2.44 jam total

Menguji medium
  parameter terlatih : 9,437,184 (1.22%)
  pita Mel           : 80
  puncak memori      : 7.56 GB
  detik per langkah  : 13.216
  taksiran waktu     : 1.27 jam/epoch, 44.59 jam total

Menguji large-v3-turbo
  parameter terlatih : 6,553,600 (0.804%)
  pita Mel           : 128
  puncak memori      : 9.2 GB
  detik per langkah  : 24.77
  taksiran waktu     : 2.39 jam/epoch, 83.56 jam total

Menguji large-v3
  M

In [7]:
# Tabel rekap
print(f"{'Varian':<16}{'Status':<14}{'Par. terlatih':>14}{'Memori':>9}{'dtk/lgk':>9}{'Jam total':>11}")
print("-" * 74)
for h in HASIL:
    if h["status"] == "BERHASIL":
        print(f"{h['varian']:<16}{h['status']:<14}{h['parameter_terlatih']:>13,}"
              f"{h['puncak_memori_gb']:>8.1f}G{h['detik_per_langkah']:>9.2f}{h['jam_total']:>11.1f}")
    else:
        print(f"{h['varian']:<16}{h['status']:<14}{'-':>14}{'-':>9}{'-':>9}{'-':>11}")

layak = [h for h in HASIL if h["status"] == "BERHASIL"]
if layak:
    terbesar = layak[-1]
    print(f"\nVarian terbesar yang dapat dilatih di mesin ini: {terbesar['varian']}")
    print(f"Perkiraan durasi pelatihan penuh: {terbesar['jam_total']} jam")
    if terbesar["jam_total"] > 24:
        print("\n[!] Lebih dari sehari penuh. Pertimbangkan Bagian 6, kurangi jumlah epoch,")
        print("    atau pindahkan pelatihan ke GPU sewaan dan laporkan sebagai tuntutan sumber daya.")
else:
    print("\nTidak ada varian yang berhasil. Lanjutkan ke Bagian 6.")

Varian          Status         Par. terlatih   Memori  dtk/lgk  Jam total
--------------------------------------------------------------------------
tiny            BERHASIL            589,824     0.8G     0.12        0.4
base            BERHASIL          1,179,648     1.2G     0.23        0.8
small           BERHASIL          3,538,944     3.0G     0.72        2.4
medium          BERHASIL          9,437,184     7.6G    13.22       44.6
large-v3-turbo  BERHASIL          6,553,600     9.2G    24.77       83.6
large-v3        MEMORI HABIS               -        -        -          -

Varian terbesar yang dapat dilatih di mesin ini: large-v3-turbo
Perkiraan durasi pelatihan penuh: 83.56 jam

[!] Lebih dari sehari penuh. Pertimbangkan Bagian 6, kurangi jumlah epoch,
    atau pindahkan pelatihan ke GPU sewaan dan laporkan sebagai tuntutan sumber daya.


## Bagian 6 — Penanganan bila memori habis (opsional)

Tiga cara menekan pemakaian memori, diurut dari yang paling kecil dampaknya terhadap hasil:

1. **Gradient checkpointing** — aktivasi dihitung ulang saat mundur alih-alih disimpan. Memori turun tajam, waktu per langkah naik sekitar 20 sampai 40 persen. Tidak mengubah hasil pelatihan.
2. **Batch diperkecil dengan akumulasi gradien** — batch efektif tetap, jadi hasilnya setara. Waktu total bertambah.
3. **Pengoptimal 8-bit** — hanya memampatkan status pengoptimal. Untuk LoRA, penghematannya kecil karena parameter terlatihnya memang sedikit.

Untuk penelitianmu, nomor 1 dan 2 aman dipakai dan cukup disebutkan di Bab 3. Keduanya tidak mengubah hasil, hanya memindahkan beban dari memori ke waktu.

In [ ]:
VARIAN_DIUJI_ULANG = "large-v3-turbo"   # ganti sesuai varian yang gagal
model_id = dict(VARIAN_DIUJI)[VARIAN_DIUJI_ULANG]

skenario = [
    {"batch": BATCH_SIZE, "grad_ckpt": True,  "optim8bit": False},
    {"batch": 2,          "grad_ckpt": True,  "optim8bit": False},
    {"batch": 1,          "grad_ckpt": True,  "optim8bit": False},
    {"batch": 1,          "grad_ckpt": True,  "optim8bit": True},
]

print(f"Menguji ulang {VARIAN_DIUJI_ULANG} dengan berbagai penanganan\n")
print(f"{'Batch':>6}{'GradCkpt':>10}{'Optim8bit':>11}{'Status':>14}{'Memori':>9}{'dtk/lgk':>9}")
print("-" * 60)

HASIL_PENANGANAN = []
for s in skenario:
    h = uji_varian(VARIAN_DIUJI_ULANG, model_id, **s)
    HASIL_PENANGANAN.append(h)
    if h["status"] == "BERHASIL":
        akum = max(1, (BATCH_SIZE * GRAD_ACCUM) // s["batch"])
        lintasan = math.ceil(n_segmen / s["batch"]) * EPOCH
        h["jam_total"] = round(h["detik_per_langkah"] * lintasan / 3600, 2)
        h["akumulasi_gradien"] = akum
        print(f"{s['batch']:>6}{str(s['grad_ckpt']):>10}{str(s['optim8bit']):>11}"
              f"{h['status']:>14}{h['puncak_memori_gb']:>8.1f}G{h['detik_per_langkah']:>9.2f}"
              f"   ~{h['jam_total']} jam")
        break
    else:
        print(f"{s['batch']:>6}{str(s['grad_ckpt']):>10}{str(s['optim8bit']):>11}{h['status']:>14}{'-':>9}{'-':>9}")

Menguji ulang large-v3-turbo dengan berbagai penanganan

 Batch  GradCkpt  Optim8bit        Status   Memori  dtk/lgk
------------------------------------------------------------
     4      True      False      BERHASIL     9.2G    23.09   ~77.88 jam


## Bagian 7 — Pemeriksaan cepat inferensi (opsional)

Bagian ini tidak menyangkut pelatihan, tetapi murah dan langsung berguna untuk subbab 3.8.3 dan 3.9.1. Isinya pengukuran RTF kasar memakai `faster-whisper` pada satu berkas audio milikmu.

Angka di sini **belum boleh dipakai di Bab 4**. Prosedur resminya menuntut jalan pemanasan yang dibuang, pengulangan beberapa kali, dan pelaporan simpangan baku. Yang ini hanya untuk memastikan pipelinenya jalan dan memberi bayangan besaran.

In [9]:
# !pip install -q faster-whisper

BERKAS_AUDIO = "contoh.wav"   # isi dengan salah satu rekamanmu, 16 kHz mono

try:
    from faster_whisper import WhisperModel
    import os

    if not os.path.exists(BERKAS_AUDIO):
        print(f"Berkas {BERKAS_AUDIO} tidak ditemukan. Lewati bagian ini.")
    else:
        tipe_hitung = "float16" if PERANGKAT == "cuda" else "int8"
        for nama in [h["varian"] for h in HASIL if h["status"] == "BERHASIL"]:
            m = WhisperModel(nama, device=PERANGKAT, compute_type=tipe_hitung)

            t0 = time.time()
            segmen, info = m.transcribe(
                BERKAS_AUDIO,
                language="id",
                task="transcribe",
                beam_size=5,
                vad_filter=True,
                condition_on_previous_text=False,
            )
            segmen = list(segmen)            # generator, transkripsi baru jalan di sini
            waktu_proses = time.time() - t0

            # Penyebut memakai durasi rekaman utuh, bukan durasi setelah VAD,
            # sesuai keputusan pada subbab 3.8.3.
            rtf = waktu_proses / info.duration
            print(f"{nama:<16} durasi {info.duration:7.1f} dtk | "
                  f"setelah VAD {info.duration_after_vad:7.1f} dtk | "
                  f"proses {waktu_proses:6.1f} dtk | RTF {rtf:.3f}"
                  f"{'  (memenuhi syarat waktu nyata)' if rtf <= 1 else '  (melebihi 1)'}")
            del m; bersihkan()

except ImportError:
    print("faster-whisper belum terpasang. Hapus komentar pada baris pip di atas.")

Berkas contoh.wav tidak ditemukan. Lewati bagian ini.


## Bagian 8 — Cara membaca hasilnya untuk Bab 3

Tiga angka dari notebook ini yang masuk ke naskah:

**Ke subbab 3.7.4 (rancangan Tahap 3).** Jumlah parameter terlatih dalam nilai absolut dan persentase, ukuran batch, dan status gradient checkpointing. Kalau kamu memakai batch lebih kecil daripada 4 yang dipakai Liu, Yang dan Qu (2024), sebutkan nilainya beserta akumulasi gradien yang menjaga batch efektif tetap sama.

**Ke subbab 3.8.4 (pencatatan tuntutan sumber daya).** Waktu pelatihan dan puncak pemakaian memori. Inilah isi jawaban paruh sumber daya pada rumusan masalah 3. Liu, Yang dan Qu (2024) melaporkan 2,6 jam pada satu GPU kelas server untuk 12 jam ujaran; angkamu adalah pembandingnya.

**Ke subbab 3.9.1 (perangkat keras).** Spesifikasi dari Bagian 1, ditambah bukti terukur bahwa perangkat itu sanggup atau tidak sanggup menjalankan pelatihan.

### Tiga kemungkinan hasil dan konsekuensinya

**Varian terpilih dapat dilatih lokal dalam waktu wajar.** Kasus paling sederhana. Pelatihan dan pengukuran di satu mesin, tidak ada yang perlu dijelaskan panjang.

**Dapat dilatih tetapi terlalu lama, misalnya di atas 24 jam.** Turunkan jumlah epoch dan pilih checkpoint berdasarkan WER validasi, atau pindahkan pelatihan ke GPU sewaan. Keduanya sah. Yang penting angka waktu ini tetap dilaporkan, sebab justru itu isi perbandingan sumber daya.

**Tidak dapat dilatih lokal sama sekali.** Ini bukan kegagalan penelitian. Batasan Masalah 5 hanya mengikat **pengukuran** pada perangkat kelas konsumen, bukan pelatihan. Latih di cloud, ukur di lokal, lalu nyatakan pemisahan itu terang-terangan di subbab 3.9.1. Temuan bahwa satu strategi adaptasi menuntut perangkat di luar jangkauan kampus biasa justru memperkuat pembahasan rumusan masalah 3.

### Yang perlu diverifikasi kalau pelatihan dilakukan di cloud

Model hasil pelatihan yang dipindahkan ke perangkat lokal harus dipastikan menghasilkan transkrip yang setara, sebab perbedaan presisi antarperangkat dapat menggeser keluaran. Jalankan satu berkas audio yang sama di kedua tempat dan bandingkan hasilnya sebelum pengukuran resmi dimulai.